# 09 — DistilBERT: Safety-Preserved Corrective Fine-Tuning

**Research project:** Explainability-Guided Feature Analysis for Reducing False Positives in Phishing Email Detection Without Increasing False Negatives

This notebook describes the corrective fine-tuning of the official DistilBERT baseline. The method added SHAP-guided legitimate-email examples and selected phishing examples to the original training records. The aim was to reduce false-positive predictions while protecting the model’s ability to identify phishing emails.

## 1. Research objective

The official DistilBERT baseline produced **31 false positives** and **22 false negatives** on the fixed test set. The research objective was to reduce false positives without increasing false negatives.

The preceding candidate-selection experiment retained **21 synthetic legitimate emails** representing **six false-positive contexts**. For the additional training stage, **21 correctly classified phishing emails** were selected from the original training set as preservation examples. The saved official baseline was the starting point for fine-tuning and was kept separately from the updated model.

## 2. Dataset and classification terminology

**Table 9.1. Dataset and baseline summary**

| Dataset measure | Number of emails |
|:---|---:|
| Cleaned dataset | 53,577 |
| Original training set | 42,861 |
| Fixed test set | 10,716 |
| Legitimate emails in the training set (class 0) | 18,991 |
| Phishing emails in the training set (class 1) | 23,870 |
| Baseline false positives | 31 |
| Baseline false negatives | 22 |

Class **0** represents legitimate emails, and class **1** represents phishing emails. The project uses the following classification terminology: **TP** denotes correctly classified legitimate emails; **TN** denotes correctly classified phishing emails; **FP** denotes legitimate emails classified as phishing; and **FN** denotes phishing emails classified as legitimate.

## 3. Construction of the corrective training dataset

All **42,861 original training records** were retained. The experiment then added 21 synthetic legitimate-email records and 21 phishing-preservation records, giving a final training set of **42,903 records**.

**Table 9.2. Composition of the corrective training dataset**

| Training component | Legitimate (class 0) | Phishing (class 1) | Total records |
|:---|---:|---:|---:|
| Original training set | 18,991 | 23,870 | 42,861 |
| Synthetic legitimate examples | 21 | 0 | 21 |
| Phishing-preservation examples | 0 | 21 | 21 |
| **Final training set** | **19,012** | **23,891** | **42,903** |

The 21 legitimate examples came from the preceding SHAP-guided selection procedure. The 21 phishing-preservation examples were selected from the original training data and added again as supplementary training records. Thus, **42,903 refers to training records, not 42,903 distinct original emails**. The original test emails were not added as training records.

## 4. Fine-tuning configuration

The saved DistilBERT baseline was fine-tuned for **one additional epoch**. The learning rate was reduced from **2 × 10⁻⁵** in the original baseline training to **1 × 10⁻⁵** for corrective fine-tuning.

**Table 9.3. Corrective fine-tuning settings**

| Setting | Value |
|:---|:---|
| Starting checkpoint | Official DistilBERT baseline |
| Maximum input length | 256 tokens |
| Training batch size | 8 |
| Additional epochs | 1 |
| Learning rate | 1 × 10⁻⁵ (0.00001) |
| Optimizer | AdamW |
| Weight decay | 0.01 |
| Gradient clipping | 1.0 |
| Warm-up steps | 0 |
| Random seed | 42 |
| Classification threshold | 0.50 |
| Trainable model parameters | 66,955,010 |
| Hardware | NVIDIA T4 GPU |

All model parameters remained trainable. The classification threshold remained **0.50**, so the intervention relied on additional model training rather than changing the prediction threshold.

## 5. Corrective fine-tuning procedure

The recorded training procedure consisted of the following stages:

1. Load the saved official DistilBERT baseline and the original training split.
2. Confirm the training-set size and class distribution.
3. Load the 21 selected synthetic legitimate-email examples.
4. Select 21 correctly classified phishing emails from the original training set for preservation during fine-tuning.
5. Add the 42 supplementary records to the original training data.
6. Fine-tune DistilBERT for one additional epoch using the settings in Table 9.3.
7. Save the updated model separately from the official baseline.

The phishing-preservation examples were included to reinforce correct phishing classification while the model learned from the new legitimate-email examples. The updated checkpoint was then available for the final evaluation.

## 6. Numerical verification of the training dataset

The following code checks the arithmetic behind Table 9.2. It verifies the reported record counts without repeating the GPU training.

In [ ]:
original_legitimate = 18_991
original_phishing = 23_870
corrective_legitimate = 21
preservation_phishing = 21

original_total = original_legitimate + original_phishing
final_legitimate = original_legitimate + corrective_legitimate
final_phishing = original_phishing + preservation_phishing
final_total = final_legitimate + final_phishing

assert original_total == 42_861
assert final_legitimate == 19_012
assert final_phishing == 23_891
assert final_total == 42_903
assert final_total - original_total == 42

print(f"Original training records: {original_total:,}")
print(f"Additional legitimate records: {corrective_legitimate:,}")
print(f"Additional phishing records: {preservation_phishing:,}")
print(f"Final training records: {final_total:,}")
print("Training-data composition checks passed.")

## 7. Interpretation

The corrective training set combined two purposes. The 21 synthetic legitimate examples addressed patterns identified during the false-positive analysis, while the 21 phishing-preservation examples reinforced examples of correctly recognised phishing emails.

This design kept the original training data intact and made a small, targeted addition of **42 training records**. The effectiveness of the updated model was assessed separately by comparing its false positives and false negatives with those of the official baseline.

## 8. Conclusion

The corrective fine-tuning experiment combined **42,861 original training records** with **21 synthetic legitimate examples** and **21 additional phishing-preservation records**, producing **42,903 training records**. DistilBERT was then fine-tuned from its official baseline checkpoint for **one additional epoch** at a learning rate of **1 × 10⁻⁵**.

The experiment produced an updated model for final evaluation. Its effect on false positives and false negatives is presented in the following evaluation notebook.